In [2]:
import jax
import jax.numpy as jnp
from aqt.jax.v2.aqt_dot_general import dot_general_make
from aqt.jax.v2.aqt_quantizer import make_fake_quant, quantizer_make, Quantizer
from aqt.jax.v2.config import fully_quantized
from jax import custom_jvp

In [3]:
def make_fake_quant(quantizer: Quantizer, calibration_axes=None):
    @custom_jvp
    def fake_quant(x):
        x_q, _ = quantizer.quant(x, calibration_axes=calibration_axes)
        return x_q.dequant()

    @fake_quant.defjvp
    def fake_quant_jvp(primals, tangents):
        (x,) = primals
        (x_dot,) = tangents
        x_q, grad_fn = quantizer.quant(x, calibration_axes=calibration_axes)
        primal_out = x_q.dequant()
        tangent_out = grad_fn(x_dot)[0]
        return primal_out, tangent_out

    return fake_quant

In [10]:
dot_general = dot_general_make(lhs_bits=3)
# dot_general = fully_quantized(fwd_bits=3, bwd_bits=None)
quantizer = quantizer_make(n_bits=3)
fake_quant = make_fake_quant(quantizer, calibration_axes=(0, 1))

x = jax.random.uniform(jax.random.key(0), (5, 3))
y = jax.random.uniform(jax.random.key(1), (5,))
# y = jnp.ones((5,))

In [15]:
@jax.value_and_grad
def loss_fn(w, x):
    y = dot_general(w, x, dimension_numbers=(((0,), (0,)), ((), ())))
    return y.sum()


@jax.value_and_grad
def loss_fn_q(w, x):
    y = fake_quant(w.T) @ x
    return y.sum()


@jax.value_and_grad
def loss_fn_full(w, x):
    y = w.T @ x
    return y.sum()


print(loss_fn(x, y))
print(loss_fn_q(x, y))
print(loss_fn_full(x, y))

(Array(3.7650957, dtype=float32), Array([[0.83079386, 0.83079386, 0.83079386],
       [0.1660409 , 0.1660409 , 0.1660409 ],
       [0.45611215, 0.45611215, 0.45611215],
       [0.6428884 , 0.6428884 , 0.6428884 ],
       [0.56865394, 0.56865394, 0.56865394]], dtype=float32))
(Array(3.932447, dtype=float32), Array([[0.83079386, 0.83079386, 0.83079386],
       [0.1660409 , 0.1660409 , 0.1660409 ],
       [0.45611215, 0.45611215, 0.45611215],
       [0.6428884 , 0.6428884 , 0.6428884 ],
       [0.56865394, 0.56865394, 0.56865394]], dtype=float32))
(Array(4.055952, dtype=float32), Array([[0.83079386, 0.83079386, 0.83079386],
       [0.1660409 , 0.1660409 , 0.1660409 ],
       [0.45611215, 0.45611215, 0.45611215],
       [0.6428884 , 0.6428884 , 0.6428884 ],
       [0.56865394, 0.56865394, 0.56865394]], dtype=float32))
